# Notebook 5: Feature Engineering & Merging

This notebook combines all extracted features and engineers derived features to create final ML-ready datasets.

## Objectives

1. **Merge Features**: Combine TerraClimate, Landsat, elevation, and land cover features
2. **Engineer Features**: Create temporal lags, interactions, and transformations
3. **Handle Missing Values**: Impute using training statistics only (no data leakage)
4. **Save ML-Ready Datasets**: Export train, validation, and test sets

## Input Files
- `train70.csv`, `val10.csv`, `test20.csv` (base data with targets)
- `terraclimate_features.csv` (climate variables)
- `landsat_features.csv` (satellite bands & indices)
- `elevation_features.csv` (elevation data)
- `land_cover_features.csv` (land cover classifications)

## Output Files
- `ml_ready_train.csv` (6,598 rows)
- `ml_ready_val.csv` (895 rows)
- `ml_ready_test.csv` (1,826 rows)

Cell 2: Imports and Constants

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import os
from datetime import datetime

# Constants
RANDOM_STATE = 42
TARGETS = ["Total Alkalinity", "Electrical Conductance", "Dissolved Reactive Phosphorus"]
MERGE_KEYS_TEMPORAL = ["Latitude", "Longitude", "Sample Date"]
MERGE_KEYS_STATIC = ["Latitude", "Longitude"]

# Variables for lag features
LAG_VARIABLES = ["ppt", "q", "soil", "tmax"]

print("Libraries loaded.")
print(f"Random state: {RANDOM_STATE}")
print(f"Targets: {TARGETS}")

Cell 3: Load input files

In [ ]:
# Load base files (with targets)
train_base = pd.read_csv("train70.csv")
val_base = pd.read_csv("val10.csv")
test_base = pd.read_csv("test20.csv")

print("Base files loaded:")
print(f"  Train: {train_base.shape}")
print(f"  Val:   {val_base.shape}")
print(f"  Test:  {test_base.shape}")

# Load feature files
terraclimate = pd.read_csv("terraclimate_features.csv")
landsat = pd.read_csv("landsat_features.csv")
elevation = pd.read_csv("elevation_features.csv")
land_cover = pd.read_csv("land_cover_features.csv")

print("\nFeature files loaded:")
print(f"  TerraClimate: {terraclimate.shape}")
print(f"  Landsat:      {landsat.shape}")
print(f"  Elevation:    {elevation.shape}")
print(f"  Land Cover:   {land_cover.shape}")

# Verify expected columns
print("\nBase file columns:", train_base.columns.tolist())
print("TerraClimate columns:", terraclimate.columns.tolist()[:5], "...")
print("Landsat columns:", landsat.columns.tolist()[:5], "...")

Cell 4: Normalize Dates and Coordiantes

In [ ]:
def normalize_date_and_coords(df):
    """
    Parse Sample Date to consistent DD-MM-YYYY format and round coordinates.
    Handles both time-varying (with dates) and static (without dates) features.
    """
    out = df.copy()
    
    # Only normalize dates if Sample Date column exists
    if "Sample Date" in out.columns:
        original_dates = out["Sample Date"].copy()
        dates = pd.to_datetime(original_dates, dayfirst=True, errors="coerce")
        parsed_str = dates.dt.strftime("%d-%m-%Y")
        out["Sample Date"] = parsed_str.where(dates.notna(), original_dates)
    
    # Always round coordinates
    out["Latitude"] = out["Latitude"].round(6)
    out["Longitude"] = out["Longitude"].round(6)
    
    return out

# Normalize all dataframes (function auto-detects if Sample Date exists)
train_base = normalize_date_and_coords(train_base)
val_base = normalize_date_and_coords(val_base)
test_base = normalize_date_and_coords(test_base)
terraclimate = normalize_date_and_coords(terraclimate)
landsat = normalize_date_and_coords(landsat)
elevation = normalize_date_and_coords(elevation)
land_cover = normalize_date_and_coords(land_cover)

print("✓ Dates normalized to DD-MM-YYYY format (where applicable).")
print("✓ Coordinates rounded to 6 decimals.")

Cell 5: Merge Features

In [ ]:
def merge_temporal_features(base_df, feature_df, feature_name):
    """Merge time-varying features on (Lat, Lon, Date)."""
    before_rows = len(base_df)
    
    # Get feature columns (exclude merge keys)
    feature_cols = [c for c in feature_df.columns if c not in MERGE_KEYS_TEMPORAL]
    
    merged = base_df.merge(
        feature_df[MERGE_KEYS_TEMPORAL + feature_cols],
        on=MERGE_KEYS_TEMPORAL,
        how="left"
    )
    
    after_rows = len(merged)
    assert before_rows == after_rows, f"{feature_name} merge changed row count: {before_rows} -> {after_rows}"
    
    print(f"  {feature_name}: {before_rows} rows -> {after_rows} rows ✓")
    return merged

def merge_static_features(base_df, feature_df, feature_name):
    """Merge static features on (Lat, Lon) only."""
    before_rows = len(base_df)
    
    # Get feature columns (exclude merge keys)
    feature_cols = [c for c in feature_df.columns if c not in MERGE_KEYS_STATIC]
    
    merged = base_df.merge(
        feature_df[MERGE_KEYS_STATIC + feature_cols],
        on=MERGE_KEYS_STATIC,
        how="left"
    )
    
    after_rows = len(merged)
    assert before_rows == after_rows, f"{feature_name} merge changed row count: {before_rows} -> {after_rows}"
    
    print(f"  {feature_name}: {before_rows} rows -> {after_rows} rows ✓")
    return merged

# Merge features for TRAIN
print("\nMerging features - TRAIN:")
train_df = train_base.copy()
train_df = merge_temporal_features(train_df, terraclimate, "TerraClimate")
train_df = merge_temporal_features(train_df, landsat, "Landsat")
train_df = merge_static_features(train_df, elevation, "Elevation")
train_df = merge_static_features(train_df, land_cover, "Land Cover")
print(f"Final train shape: {train_df.shape}")

# Merge features for VAL
print("\nMerging features - VAL:")
val_df = val_base.copy()
val_df = merge_temporal_features(val_df, terraclimate, "TerraClimate")
val_df = merge_temporal_features(val_df, landsat, "Landsat")
val_df = merge_static_features(val_df, elevation, "Elevation")
val_df = merge_static_features(val_df, land_cover, "Land Cover")
print(f"Final val shape: {val_df.shape}")

# Merge features for TEST
print("\nMerging features - TEST:")
test_df = test_base.copy()
test_df = merge_temporal_features(test_df, terraclimate, "TerraClimate")
test_df = merge_temporal_features(test_df, landsat, "Landsat")
test_df = merge_static_features(test_df, elevation, "Elevation")
test_df = merge_static_features(test_df, land_cover, "Land Cover")
print(f"Final test shape: {test_df.shape}")

Cell 6: Create Location_ID (need it to calculate temporal features)

Cell 7: Compute Lag and Rolling Features

In [ ]:
def compute_lag_features(df, variables=LAG_VARIABLES):
    """
    Compute lag and rolling features for specified variables.
    CRITICAL: This must be done per split to avoid data leakage.
    """
    df = df.copy()
    
    # Parse dates for sorting
    df["_date_parsed"] = pd.to_datetime(df["Sample Date"], dayfirst=True, errors="coerce")
    
    # Sort by location and date
    df = df.sort_values(["Location_ID", "_date_parsed"]).reset_index(drop=True)
    
    for var in variables:
        if var not in df.columns:
            print(f"  Warning: {var} not found in dataframe, skipping")
            continue
        
        # Group by location for temporal operations
        grouped = df.groupby("Location_ID")[var]
        
        # Lag features
        df[f"{var}_lag1"] = grouped.shift(1)
        df[f"{var}_lag3"] = grouped.shift(3)
        
        # Rolling features
        df[f"{var}_roll3_mean"] = grouped.rolling(window=3, min_periods=1).mean().reset_index(level=0, drop=True)
        df[f"{var}_roll6_mean"] = grouped.rolling(window=6, min_periods=1).mean().reset_index(level=0, drop=True)
    
    # Drop temporary date column
    df = df.drop(columns=["_date_parsed"])
    
    n_lag_features = len(variables) * 4  # 4 features per variable
    print(f"  Created {n_lag_features} lag/rolling features")
    
    return df

print("Computing lag & rolling features (per split to avoid leakage):")
print("\nTRAIN:")
train_df = compute_lag_features(train_df)

print("\nVAL:")
val_df = compute_lag_features(val_df)

print("\nTEST:")
test_df = compute_lag_features(test_df)

print("\nLag features created for variables:", LAG_VARIABLES)

Cell 8: Computer interaction terms

In [ ]:
def create_interaction_features(df):
    """Create interaction and ratio features."""
    df = df.copy()
    
    # Water balance
    if "ppt" in df.columns and "pet" in df.columns:
        df["water_balance"] = df["ppt"] - df["pet"]
    
    # Temperature range
    if "tmax" in df.columns and "tmin" in df.columns:
        df["temp_range"] = df["tmax"] - df["tmin"]
    
    # Runoff ratio (handle division by zero)
    if "q" in df.columns and "ppt" in df.columns:
        df["runoff_ratio"] = np.where(df["ppt"] > 0, df["q"] / df["ppt"], 0)
    
    # Moisture stress
    if "pet" in df.columns and "aet" in df.columns:
        df["moisture_stress"] = df["pet"] - df["aet"]
    
    # NDVI interactions
    if "NDVI" in df.columns and "ppt" in df.columns:
        df["NDVI_x_ppt"] = df["NDVI"] * df["ppt"]
    
    if "NDVI" in df.columns and "soil" in df.columns:
        df["NDVI_x_soil"] = df["NDVI"] * df["soil"]
    
    # Count created features
    interaction_features = [
        "water_balance", "temp_range", "runoff_ratio", 
        "moisture_stress", "NDVI_x_ppt", "NDVI_x_soil"
    ]
    created = [f for f in interaction_features if f in df.columns]
    print(f"  Created {len(created)} interaction features: {created}")
    
    return df

print("Creating interaction features:")
train_df = create_interaction_features(train_df)
val_df = create_interaction_features(val_df)
test_df = create_interaction_features(test_df)

Cell 9: Create temporal features

In [ ]:
def create_temporal_features(df):
    """Extract and encode temporal features from Sample Date."""
    df = df.copy()
    
    # Parse date
    dates = pd.to_datetime(df["Sample Date"], dayfirst=True, errors="coerce")
    
    # Extract components
    df["year"] = dates.dt.year
    df["month"] = dates.dt.month
    
    # Cyclical encoding for month
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    
    # Season (Southern Hemisphere)
    def get_season(month):
        if pd.isna(month):
            return None
        if month in [12, 1, 2]:
            return "Summer"
        elif month in [3, 4, 5]:
            return "Autumn"
        elif month in [6, 7, 8]:
            return "Winter"
        else:  # 9, 10, 11
            return "Spring"
    
    df["season"] = df["month"].apply(get_season)
    
    # Wet season indicator (Oct-Mar in Southern Hemisphere)
    df["is_wet_season"] = df["month"].isin([10, 11, 12, 1, 2, 3]).astype(int)
    
    temporal_features = ["year", "month", "month_sin", "month_cos", "season", "is_wet_season"]
    print(f"  Created {len(temporal_features)} temporal features: {temporal_features}")
    
    return df

print("Creating temporal features:")
train_df = create_temporal_features(train_df)
val_df = create_temporal_features(val_df)
test_df = create_temporal_features(test_df)

print("\nSeason distribution (train):")
print(train_df["season"].value_counts())

Cell 10: Create target transforms

In [ ]:
def create_target_transforms(df):
    """Create log-transformed targets (for training purposes)."""
    df = df.copy()
    
    # Log transforms (only if targets exist)
    if "Total Alkalinity" in df.columns:
        df["log_Alkalinity"] = np.log1p(df["Total Alkalinity"])
        print("  Created log_Alkalinity")
    
    if "Dissolved Reactive Phosphorus" in df.columns:
        df["log_DRP"] = np.log1p(df["Dissolved Reactive Phosphorus"])
        print("  Created log_DRP")
    
    return df

print("Creating target transforms:")
print("\nTRAIN (has targets):")
train_df = create_target_transforms(train_df)

print("\nVAL (has targets):")
val_df = create_target_transforms(val_df)

print("\nTEST (has targets):")
test_df = create_target_transforms(test_df)

Cell 11: Create Landsat Missing Indicator

In [ ]:
def create_missing_indicator(df):
    """Create indicator for missing Landsat data (before imputation)."""
    df = df.copy()
    
    if "nir" in df.columns:
        df["landsat_missing"] = df["nir"].isnull().astype(int)
        n_missing = df["landsat_missing"].sum()
        pct_missing = 100 * n_missing / len(df)
        print(f"  landsat_missing: {n_missing} rows ({pct_missing:.1f}%)")
    else:
        df["landsat_missing"] = 0
        print("  nir column not found, set landsat_missing = 0")
    
    return df

print("Creating Landsat missing indicator (BEFORE imputation):")
train_df = create_missing_indicator(train_df)
val_df = create_missing_indicator(val_df)
test_df = create_missing_indicator(test_df)

Cell 12: One Hot encode Land cover and season (need dummy variables for these categories)

In [ ]:
def encode_categorical_features(train, val, test):
    """
    One-hot encode categorical features.
    - Land cover: drop_first=True (avoid multicollinearity)
    - Season: drop_first=True (avoid multicollinearity)
    """
    
    # Combine to get all categories
    all_data = pd.concat([train, val, test], ignore_index=True)
    
    encoded_dfs = {"train": train.copy(), "val": val.copy(), "test": test.copy()}
    
    # ========================================
    # 1. ENCODE LAND COVER
    # ========================================
    for col in ["land_cover_250m", "land_cover_500m"]:
        if col not in all_data.columns:
            print(f"  {col} not found, skipping")
            continue
        
        # Get dummies
        dummies = pd.get_dummies(all_data[col], prefix=col, drop_first=True, dtype=float)
        
        # Split back into train/val/test
        train_len = len(train)
        val_len = len(val)
        
        train_dummies = dummies.iloc[:train_len].reset_index(drop=True)
        val_dummies = dummies.iloc[train_len:train_len+val_len].reset_index(drop=True)
        test_dummies = dummies.iloc[train_len+val_len:].reset_index(drop=True)
        
        # Add to dataframes
        encoded_dfs["train"] = pd.concat([encoded_dfs["train"].reset_index(drop=True), train_dummies], axis=1)
        encoded_dfs["val"] = pd.concat([encoded_dfs["val"].reset_index(drop=True), val_dummies], axis=1)
        encoded_dfs["test"] = pd.concat([encoded_dfs["test"].reset_index(drop=True), test_dummies], axis=1)
        
        # Drop original column
        encoded_dfs["train"] = encoded_dfs["train"].drop(columns=[col])
        encoded_dfs["val"] = encoded_dfs["val"].drop(columns=[col])
        encoded_dfs["test"] = encoded_dfs["test"].drop(columns=[col])
        
        print(f"  {col}: created {len(dummies.columns)} dummy variables (dropped first category)")
    
    # ========================================
    # 2. ENCODE SEASON
    # ========================================
    if "season" in all_data.columns:
        # Get dummies for season
        season_dummies = pd.get_dummies(all_data["season"], prefix="season", drop_first=True, dtype=float)
        
        # Split back into train/val/test
        train_len = len(train)
        val_len = len(val)
        
        train_season = season_dummies.iloc[:train_len].reset_index(drop=True)
        val_season = season_dummies.iloc[train_len:train_len+val_len].reset_index(drop=True)
        test_season = season_dummies.iloc[train_len+val_len:].reset_index(drop=True)
        
        # Add to dataframes
        encoded_dfs["train"] = pd.concat([encoded_dfs["train"].reset_index(drop=True), train_season], axis=1)
        encoded_dfs["val"] = pd.concat([encoded_dfs["val"].reset_index(drop=True), val_season], axis=1)
        encoded_dfs["test"] = pd.concat([encoded_dfs["test"].reset_index(drop=True), test_season], axis=1)
        
        # Drop original season column
        encoded_dfs["train"] = encoded_dfs["train"].drop(columns=["season"])
        encoded_dfs["val"] = encoded_dfs["val"].drop(columns=["season"])
        encoded_dfs["test"] = encoded_dfs["test"].drop(columns=["season"])
        
        print(f"  season: created {len(season_dummies.columns)} dummy variables (dropped first category)")
        print(f"    Categories: {sorted(season_dummies.columns.tolist())}")
    
    return encoded_dfs["train"], encoded_dfs["val"], encoded_dfs["test"]

print("One-hot encoding categorical features:")
train_df, val_df, test_df = encode_categorical_features(train_df, val_df, test_df)

print(f"\nShapes after encoding:")
print(f"  Train: {train_df.shape}")
print(f"  Val:   {val_df.shape}")
print(f"  Test:  {test_df.shape}")

Cell 12: Handle Missing Values and Infinites

In [ ]:
def handle_missing_and_inf(train, val, test):
    """
    Handle missing values and infinite values.
    - Compute imputation values from TRAIN only
    - Apply to val and test (no data leakage)
    """
    train = train.copy()
    val = val.copy()
    test = test.copy()
    
    # Replace inf with NaN
    print("Replacing infinite values with NaN...")
    train.replace([np.inf, -np.inf], np.nan, inplace=True)
    val.replace([np.inf, -np.inf], np.nan, inplace=True)
    test.replace([np.inf, -np.inf], np.nan, inplace=True)
    
    # Get numeric columns (exclude targets and identifiers)
    # NOTE: "season" was removed in Cell 12 (replaced with season_* dummies)
    exclude_cols = TARGETS + ["Latitude", "Longitude", "Sample Date", "Location_ID"]
    numeric_cols = train.select_dtypes(include=[np.number]).columns
    numeric_cols = [c for c in numeric_cols if c not in exclude_cols]
    
    print(f"Computing median imputation from TRAIN for {len(numeric_cols)} numeric columns...")
    
    # Compute medians from TRAIN only
    train_medians = train[numeric_cols].median()
    
    # Impute TRAIN
    train[numeric_cols] = train[numeric_cols].fillna(train_medians)
    
    # Impute VAL using TRAIN medians
    val[numeric_cols] = val[numeric_cols].fillna(train_medians)
    
    # Impute TEST using TRAIN medians
    test[numeric_cols] = test[numeric_cols].fillna(train_medians)
    
    # Verify no NaNs remain in numeric columns
    train_nans = train[numeric_cols].isna().sum().sum()
    val_nans = val[numeric_cols].isna().sum().sum()
    test_nans = test[numeric_cols].isna().sum().sum()
    
    print(f"\nMissing values after imputation:")
    print(f"  Train: {train_nans}")
    print(f"  Val:   {val_nans}")
    print(f"  Test:  {test_nans}")
    
    if train_nans + val_nans + test_nans > 0:
        print("\n⚠️ WARNING: Some NaN values remain!")
    else:
        print("\n✓ All numeric columns imputed successfully")
    
    return train, val, test

train_df, val_df, test_df = handle_missing_and_inf(train_df, val_df, test_df)

Cell 14: Quality Checks and Validation

In [ ]:
print("=" * 80)
print("QUALITY CHECKS")
print("=" * 80)

# 1. Verify row counts
print("\n1. Row Count Verification:")
assert len(train_df) == 6598, f"Train row count mismatch: {len(train_df)} != 6598"
assert len(val_df) == 895, f"Val row count mismatch: {len(val_df)} != 895"
assert len(test_df) == 1826, f"Test row count mismatch: {len(test_df)} != 1826"
print(f"  ✓ Train: {len(train_df)} rows")
print(f"  ✓ Val:   {len(val_df)} rows")
print(f"  ✓ Test:  {len(test_df)} rows")

# 2. Check for missing values
print("\n2. Missing Values Check:")
train_missing = train_df.isna().sum().sum()
val_missing = val_df.isna().sum().sum()
test_missing = test_df.isna().sum().sum()
print(f"  Train: {train_missing} missing values")
print(f"  Val:   {val_missing} missing values")
print(f"  Test:  {test_missing} missing values")

if train_missing + val_missing + test_missing > 0:
    print("\n  Columns with missing values:")
    for df_name, df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
        missing = df.isna().sum()
        if missing.sum() > 0:
            print(f"    {df_name}:")
            print(missing[missing > 0].head(10))

# 3. Check for infinite values
print("\n3. Infinite Values Check:")
train_inf = np.isinf(train_df.select_dtypes(include=[np.number])).sum().sum()
val_inf = np.isinf(val_df.select_dtypes(include=[np.number])).sum().sum()
test_inf = np.isinf(test_df.select_dtypes(include=[np.number])).sum().sum()
print(f"  Train: {train_inf} infinite values")
print(f"  Val:   {val_inf} infinite values")
print(f"  Test:  {test_inf} infinite values")

# 4. Verify identical column sets
print("\n4. Column Set Verification:")
train_cols = set(train_df.columns)
val_cols = set(val_df.columns)
test_cols = set(test_df.columns)

if train_cols == val_cols == test_cols:
    print(f"  ✓ All datasets have identical columns ({len(train_cols)} columns)")
else:
    print("  ✗ Column sets differ!")
    print(f"    Only in train: {train_cols - val_cols - test_cols}")
    print(f"    Only in val: {val_cols - train_cols - test_cols}")
    print(f"    Only in test: {test_cols - train_cols - val_cols}")

# 5. Feature count summary
print("\n5. Feature Count Summary:")
print(f"  Total columns: {len(train_df.columns)}")
print(f"  Numeric features: {len(train_df.select_dtypes(include=[np.number]).columns)}")
print(f"  Categorical features: {len(train_df.select_dtypes(exclude=[np.number]).columns)}")

# Check for land cover encoding
land_cover_cols = [c for c in train_df.columns if c.startswith("land_cover_")]
print(f"  Land cover dummy variables: {len(land_cover_cols)}")

print("\n" + "=" * 80)

Cell 15: Save Output Files (had to separate cells for snowflake 10mb limit)

In [ ]:
import base64
from IPython.display import HTML, display

with open('ml_ready_train.csv', 'rb') as f:
    train_b64 = base64.b64encode(f.read()).decode()

html = f'<a href="data:text/csv;base64,{train_b64}" download="ml_ready_train.csv">Download ml_ready_train.csv</a>'

display(HTML(html))

In [ ]:
import base64
from IPython.display import HTML, display

with open('ml_ready_val.csv', 'rb') as f:
    val_b64 = base64.b64encode(f.read()).decode()

html = f'<a href="data:text/csv;base64,{val_b64}" download="ml_ready_val.csv">Download ml_ready_val.csv</a>'

display(HTML(html))

In [ ]:
import base64
from IPython.display import HTML, display

with open('ml_ready_test.csv', 'rb') as f:
    test_b64 = base64.b64encode(f.read()).decode()

html = f'<a href="data:text/csv;base64,{test_b64}" download="ml_ready_test.csv">Download ml_ready_test.csv</a>'

display(HTML(html))

Cell 16:Summary and Column List

In [ ]:
print("=" * 80)
print("FINAL SUMMARY")
print("=" * 80)

print("\n📊 Dataset Statistics:")
print(f"  Training:   {len(train_df):,} rows × {len(train_df.columns)} columns")
print(f"  Validation: {len(val_df):,} rows × {len(val_df.columns)} columns")
print(f"  Test:       {len(test_df):,} rows × {len(test_df.columns)} columns")

print("\n📋 Feature Categories:")

# Organize columns by category
key_cols = ["Latitude", "Longitude", "Sample Date", "Location_ID"]
target_cols = [c for c in train_df.columns if c in TARGETS or c.startswith("log_")]
terraclimate_cols = ["pet", "aet", "ppt", "tmax", "tmin", "q", "soil", "swe", "def", "pdsi", "vap", "vpd", "ws", "srad"]
landsat_cols = [c for c in train_df.columns if c in ["nir", "green", "red", "blue", "swir16", "swir22", "NDVI", "MNDWI", "NDMI", "NDTI", "Chlorophyll_Proxy", "BSI", "SAVI", "AWEI_nsh", "AWEI_sh", "NRI", "Turbidity_Index"]]
lag_cols = [c for c in train_df.columns if "_lag" in c or "_roll" in c]
interaction_cols = ["water_balance", "temp_range", "runoff_ratio", "moisture_stress", "NDVI_x_ppt", "NDVI_x_soil"]
interaction_cols = [c for c in interaction_cols if c in train_df.columns]

# UPDATED: Look for season_* columns instead of "season"
temporal_cols = ["year", "month", "month_sin", "month_cos", "is_wet_season"]
temporal_cols = [c for c in temporal_cols if c in train_df.columns]
season_cols = [c for c in train_df.columns if c.startswith("season_")]

land_cover_cols = [c for c in train_df.columns if c.startswith("land_cover_")]
other_cols = ["elevation", "landsat_missing"]
other_cols = [c for c in other_cols if c in train_df.columns]

print(f"  Key columns:        {len(key_cols)}")
print(f"  Target variables:   {len(target_cols)}")
print(f"  TerraClimate:       {len([c for c in terraclimate_cols if c in train_df.columns])}")
print(f"  Landsat:            {len(landsat_cols)}")
print(f"  Lag/Rolling:        {len(lag_cols)}")
print(f"  Interactions:       {len(interaction_cols)}")
print(f"  Temporal:           {len(temporal_cols)}")
print(f"  Season (encoded):   {len(season_cols)}")  # UPDATED
print(f"  Land cover encoded: {len(land_cover_cols)}")
print(f"  Other:              {len(other_cols)}")

print("\n📝 Complete Column List:")
for i, col in enumerate(sorted(train_df.columns), 1):
    dtype = train_df[col].dtype
    print(f"  {i:3d}. {col:40s} ({dtype})")

print("\n" + "=" * 80)
print("✓ NOTEBOOK 5 COMPLETE - ML-READY DATASETS SAVED")
print("=" * 80)

# Show first few rows
print("\nFirst 3 rows of training data:")
train_df.head(3)